In [ ]:
# ============================================================================
# CELL 1: IMPORTS AND SETUP
# ============================================================================

# LangChain and OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Geocoding and API requests
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import requests

# Data processing
import pandas as pd
import numpy as np
import json
from pathlib import Path
import time
from datetime import datetime, timedelta

# Clustering
from sklearn.cluster import DBSCAN

# Visualization
import folium
from folium.plugins import MarkerCluster

# Image processing for Instagram
import base64
from openai import OpenAI

# ============================================================================
# API KEYS
# ============================================================================
OPENAI_API_KEY = "API" # Placeholder - to be filled in
FIRMS_API_KEY = "API"  # Placeholder - to be filled in
APIFY_API_KEY = "API"  # Placeholder - to be filled in

# ============================================================================
# INITIALIZE TOOLS
# ============================================================================
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
search = DuckDuckGoSearchRun()
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
geolocator = Nominatim(user_agent="fire_tracker_app", timeout=10)

print("✓ All imports loaded successfully")
print("✓ LLM initialized: GPT-4o-mini")
print("✓ Tools ready: Search, Wikipedia, Geocoder")

In [ ]:
# ============================================================================
# CELL 2: DATA COLLECTOR AGENT
# ============================================================================

class DataCollectorAgent:
    """
    Agent responsible for collecting all raw data:
    - Fire information from web search
    - Social media data (Instagram + Twitter) via Apify
    - FIRMS satellite data
    - Creates Instagram metadata
    """
    
    def __init__(self):
        """Initialize the Data Collector Agent"""
        self.fire_info = None
        self.instagram_data = []
        self.twitter_data = []
        self.firms_data = None
        
        # Create directory structure for images and metadata
        self.base_dir = Path("raw_data")
        self.instagram_images_dir = self.base_dir / "instagram" / "images"
        self.instagram_metadata_dir = self.base_dir / "instagram" / "metadata"
        self.twitter_dir = self.base_dir / "twitter"
        self.firms_dir = self.base_dir / "firms"
        
        self._setup_directories()
    
    def _setup_directories(self):
        """Create necessary directory structure"""
        self.instagram_images_dir.mkdir(parents=True, exist_ok=True)
        self.instagram_metadata_dir.mkdir(parents=True, exist_ok=True)
        self.twitter_dir.mkdir(parents=True, exist_ok=True)
        self.firms_dir.mkdir(parents=True, exist_ok=True)
        print("✓ Directory structure created")
    
    def extract_fire_info(self, user_input):
        """
        Extract fire name from user input and search web for fire details        
        Returns:
            dict: Fire information including name, location, dates, coordinates
        """
        print(f"\n{'='*60}")
        print(f"STEP 1: Extracting Fire Information")
        print(f"{'='*60}")
        print(f"User input: {user_input}")
        
        # Prompt 1: Extract basic fire info from user input
        extraction_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a Data collector. Extract fire information from user queries.

Output ONLY valid JSON with this structure:
{{
  "fire_name": "name if mentioned, else null",
  "location": "extracted location or null",
  "dates": "any mentioned dates or null",
  "needs_search": true or false,
  "search_query": "optimized query for fire information or null"
}}
"""),
            ("user", "{user_input}")
        ])
        
        chain = extraction_prompt | llm
        response = chain.invoke({"user_input": user_input})
        
        try:
            fire_info = json.loads(response.content)
            print(f"\nExtracted info: {json.dumps(fire_info, indent=2)}")
        except json.JSONDecodeError:
            print(f"Error parsing response: {response.content}")
            return None
        
        # If we need to search for more details
        if fire_info.get("needs_search"):
            print(f"\nSearching web for fire details...")
            fire_details = self._search_fire_details(
                fire_info["search_query"], 
                fire_info.get("fire_name", "")
            )
            
            if fire_details:
                self.fire_info = fire_details
                print(f"\n✓ Fire details found:")
                print(f"  Name: {fire_details['fire_name']}")
                print(f"  Location: {fire_details['location']}")
                print(f"  Start Date: {fire_details['start_date']}")
                print(f"  Coordinates: {fire_details['lat']}, {fire_details['lon']}")
                return fire_details
        
        return None
    
    def _search_fire_details(self, search_query, fire_name):
        """
        Search for detailed fire information using DuckDuckGo and Wikipedia
        
        Args:
            search_query: Optimized search query
            fire_name: Name of the fire
        
        Returns:
            dict: Detailed fire information
        """
        # Search DuckDuckGo
        print(f"Searching DuckDuckGo for: {search_query}")
        ddg_results = search.run(search_query)
        print(f"DDG results preview: {ddg_results[:200]}...")
        
        # Search Wikipedia
        wiki_query = f"{fire_name} wildfire"
        print(f"\nSearching Wikipedia for: {wiki_query}")
        try:
            wiki_results = wikipedia.run(wiki_query)
            print(f"Wikipedia results preview: {wiki_results[:200]}...")
        except:
            wiki_results = ""
            print("No Wikipedia results found")
        
        # Combine results
        combined_results = f"DuckDuckGo:\n{ddg_results}\n\nWikipedia:\n{wiki_results}"
        
        # Prompt 2: Extract structured fire details from search results
        search_extraction_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a Data collector extracting wildfire information from search results.

Output ONLY valid JSON with this structure:
{{
  "fire_name": "official fire name",
  "location": "primary location (city, county, state)",
  "lat": latitude as number or null,
  "lon": longitude as number or null,
  "start_date": "YYYY-MM-DD or null",
  "end_date": "YYYY-MM-DD or 'ongoing' or null",
  "confidence": "high/medium/low"
}}

Extract the most accurate information from the search results. If coordinates aren't found, set them to null."""),
            ("user", "Search results:\n{search_results}\n\nExtract fire information.")
        ])
        
        chain = search_extraction_prompt | llm
        response = chain.invoke({"search_results": combined_results})
        
        try:
            fire_details = json.loads(response.content)
            
            # If coordinates not found, geocode the location
            if not fire_details.get("lat") or not fire_details.get("lon"):
                print(f"\nGeocoding location: {fire_details['location']}")
                lat, lon = self._geocode_location(fire_details['location'])
                fire_details['lat'] = lat
                fire_details['lon'] = lon
            
            return fire_details
        
        except json.JSONDecodeError:
            print(f"Error parsing search results: {response.content}")
            return None
    
    def _geocode_location(self, location_name):
        """
        Geocode a location name to lat/lon coordinates
        
        Args:
            location_name: Location string to geocode
        
        Returns:
            tuple: (latitude, longitude) or (None, None) if failed
        """
        try:
            time.sleep(1)  # Rate limiting
            location = geolocator.geocode(location_name)
            
            if location:
                print(f"✓ Geocoded: {location.latitude}, {location.longitude}")
                return location.latitude, location.longitude
            else:
                print(f"✗ Geocoding failed for: {location_name}")
                return None, None
        
        except GeocoderTimedOut:
            print(f"✗ Geocoding timed out")
            return None, None
    
    def collect_social_media(self):
        """
        Collect Instagram and Twitter data via Apify
        Downloads Instagram images and creates metadata
        
        Returns:
            tuple: (instagram_data, twitter_data)
        """
        if not self.fire_info:
            print("Error: Fire info not available. Run extract_fire_info() first.")
            return None, None
        
        print(f"\n{'='*60}")
        print(f"STEP 2: Collecting Social Media Data")
        print(f"{'='*60}")
        
        # Collect Instagram data
        self._collect_instagram()
        
        # Collect Twitter data
        self._collect_twitter()
        
        print(f"\n✓ Social media collection complete")
        print(f"  Instagram posts: {len(self.instagram_data)}")
        print(f"  Twitter posts: {len(self.twitter_data)}")
        
        return self.instagram_data, self.twitter_data
    
    def _collect_instagram(self):
        """Collect Instagram posts via Apify and download images"""
        print(f"\nCollecting Instagram data...")
        
        fire_name = self.fire_info['fire_name']
        start_date = self.fire_info['start_date']
        end_date = self.fire_info.get('end_date', 'ongoing')
        
        # Prepare search hashtags
        hashtag = f"#{fire_name.replace(' ', '')}"
        
        print(f"  Searching Instagram for: {hashtag}")
        print(f"  Date range: {start_date} to {end_date}")
        
        try:
            # Note: This code won't run with APIFY_API_KEY = "example"
            # But it's structured correctly for when real API key is used
            from apify_client import ApifyClient
            
            client = ApifyClient(APIFY_API_KEY)
            
            # Configure Instagram scraper
            run_input = {
                "directUrls": [],  # Will be populated with URLs from search
                "resultsLimit": 100,
                "searchLimit": 100,
                "searchType": "hashtag",
                "search": hashtag,
                "proxyConfiguration": {
                    "useApifyProxy": True
                }
            }
            
            # Run Apify Instagram scraper
            print(f"  Running Apify Instagram scraper...")
            run = client.actor("apify/instagram-scraper").call(run_input=run_input)
            
            # Fetch results
            for item in client.dataset(run["defaultDatasetId"]).iterate_items():
                post_url = item.get('url', '')
                
                if not post_url:
                    continue
                
                # Get full post details including image URLs
                post_details = self._get_instagram_post_details(client, post_url)
                
                if post_details:
                    # Download images and create metadata
                    self._process_instagram_post(item, post_details)
            
            print(f"  ✓ Instagram collection complete: {len(self.instagram_data)} posts")
        
        except Exception as e:
            print(f"  ⚠️ Instagram collection error: {e}")
            print(f"  (This is expected with APIFY_API_KEY = 'example')")
    
    def _get_instagram_post_details(self, client, post_url):
        """
        Get full Instagram post details including image URLs
        
        Args:
            client: ApifyClient instance
            post_url: Instagram post URL
        
        Returns:
            dict: Post details with image URLs
        """
        try:
            run_input = {
                "url": post_url,
                "proxyConfiguration": {
                    "useApifyProxy": True,
                    "apifyProxyGroups": ["RESIDENTIAL"]
                }
            }
            
            run = client.actor("oGjh5FfvNJgbjaAc1").call(run_input=run_input)
            
            # Extract image URLs
            image_urls = []
            for item in client.dataset(run["defaultDatasetId"]).iterate_items():
                if 'displayUrl' in item:
                    image_urls.append(item['displayUrl'])
                elif 'images' in item:
                    image_urls.extend(item['images'])
            
            return {'image_urls': image_urls} if image_urls else None
        
        except Exception as e:
            print(f"    ⚠️ Error fetching post details: {e}")
            return None
    
    def _process_instagram_post(self, item, post_details):
        """
        Process Instagram post: download images and create metadata
        
        Args:
            item: Instagram post data from Apify
            post_details: Full post details with image URLs
        """
        post_id = self._extract_post_id(item.get('url', ''))
        
        if not post_id:
            return
        
        caption = item.get('caption', '')
        timestamp = item.get('timestamp', '')
        location_name = item.get('locationName', '')
        location_id = item.get('locationId', '')
        
        # Extract hashtags
        hashtags, clean_caption = self._extract_hashtags(caption)
        
        # Download each image in the post
        image_urls = post_details.get('image_urls', [])
        
        for img_index, img_url in enumerate(image_urls):
            # Create filename
            if len(image_urls) == 1:
                filename = f"{post_id}.jpg"
            else:
                filename = f"{post_id}_{img_index + 1}.jpg"
            
            filepath = self.instagram_images_dir / filename
            
            # Download image
            if self._download_image(img_url, filepath):
                # Create metadata
                metadata = {
                    'image_id': post_id,
                    'image_filename': filename,
                    'image_index': img_index,
                    'caption_original': caption,
                    'caption_clean': clean_caption,
                    'hashtags': hashtags,
                    'timestamp': timestamp,
                    'location_name': location_name,
                    'location_id': location_id,
                    'instagram_url': item.get('url', ''),
                    'download_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    # Placeholder for LLM analysis
                    'llm_extracted_location': '',
                    'coordinates': [],
                    'llm_confidence': ''
                }
                
                # Save metadata JSON file
                metadata_filename = f"{post_id}"
                if img_index > 0:
                    metadata_filename += f"_{img_index}"
                metadata_filename += "_metadata.json"
                
                metadata_path = self.instagram_metadata_dir / metadata_filename
                
                with open(metadata_path, 'w', encoding='utf-8') as f:
                    json.dump(metadata, f, indent=2, ensure_ascii=False)
                
                # Store in memory
                self.instagram_data.append(metadata)
                
                print(f"    ✓ Downloaded and created metadata: {filename}")
            
            time.sleep(1)  # Rate limiting
    
    def _extract_post_id(self, url):
        """Extract post ID from Instagram URL"""
        match = re.search(r'/p/([A-Za-z0-9_-]+)', url)
        return match.group(1) if match else None
    
    def _extract_hashtags(self, caption):
        """Extract hashtags from caption text"""
        if pd.isna(caption) or caption == '':
            return [], ''
        
        import re
        hashtags = re.findall(r'#(\w+)', caption)
        clean_caption = re.sub(r'#\w+', '', caption)
        clean_caption = ' '.join(clean_caption.split())
        
        return hashtags, clean_caption
    
    def _download_image(self, url, filepath):
        """
        Download image from URL
        
        Args:
            url: Image URL
            filepath: Path to save image
        
        Returns:
            bool: True if successful, False otherwise
        """
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            
            with open(filepath, 'wb') as f:
                f.write(response.content)
            
            return True
        
        except Exception as e:
            print(f"    ✗ Download failed: {e}")
            return False
    
    def _collect_twitter(self):
        """Collect Twitter data via Apify"""
        print(f"\nCollecting Twitter data...")
        
        fire_name = self.fire_info['fire_name']
        start_date = self.fire_info['start_date']
        end_date = self.fire_info.get('end_date', 'ongoing')
        
        search_query = f"#{fire_name.replace(' ', '')} OR {fire_name}"
        
        print(f"  Searching Twitter for: {search_query}")
        print(f"  Date range: {start_date} to {end_date}")
        
        try:
            from apify_client import ApifyClient
            
            client = ApifyClient(APIFY_API_KEY)
            
            # Configure Twitter scraper
            run_input = {
                "searchTerms": [search_query],
                "maxItems": 100,
                "maxTweetsPerQuery": 100,
                "start": start_date,
                "end": end_date if end_date != 'ongoing' else None
            }
            
            # Run Apify Twitter scraper
            print(f"  Running Apify Twitter scraper...")
            run = client.actor("apify/twitter-scraper").call(run_input=run_input)
            
            # Fetch results
            for item in client.dataset(run["defaultDatasetId"]).iterate_items():
                tweet_data = {
                    'id': item.get('id', ''),
                    'text': item.get('text', ''),
                    'created_at': item.get('createdAt', ''),
                    'author': item.get('author', {}).get('userName', ''),
                    'url': item.get('url', ''),
                    'retweet_count': item.get('retweetCount', 0),
                    'like_count': item.get('likeCount', 0),
                    'location': item.get('place', {}).get('fullName', '') if item.get('place') else ''
                }
                
                self.twitter_data.append(tweet_data)
            
            print(f"  ✓ Twitter collection complete: {len(self.twitter_data)} tweets")
        
        except Exception as e:
            print(f"  ⚠️ Twitter collection error: {e}")
            print(f"  (This is expected with APIFY_API_KEY = 'example')")
    
    def collect_firms(self):
        """
        Collect FIRMS satellite fire detection data
        
        Returns:
            pandas.DataFrame: FIRMS fire detections
        """
        if not self.fire_info:
            print("Error: Fire info not available. Run extract_fire_info() first.")
            return None
        
        print(f"\n{'='*60}")
        print(f"STEP 3: Collecting FIRMS Satellite Data")
        print(f"{'='*60}")
        
        lat = self.fire_info['lat']
        lon = self.fire_info['lon']
        start_date = self.fire_info['start_date']
        
        # Try multiple satellite sources
        sources = ["VIIRS_SNPP_SP", "VIIRS_NOAA20_SP", "MODIS_SP", "GOES"]
        
        for source in sources:
            print(f"\nTrying source: {source}")
            firms_data = self._query_firms_source(lat, lon, start_date, source)
            
            if firms_data is not None and len(firms_data) > 0:
                self.firms_data = firms_data
                print(f"✓ SUCCESS with {source}: {len(firms_data)} detections")
                return firms_data
        
        print(f"\n✗ No FIRMS data found in any source")
        return None
    
    def _query_firms_source(self, lat, lon, start_date, source):
        """
        Query FIRMS API for specific satellite source
        
        Args:
            lat, lon: Fire coordinates
            start_date: Fire start date (YYYY-MM-DD)
            source: Satellite source name
        
        Returns:
            pandas.DataFrame: Fire detections or None
        """
        # Create bounding box (±0.5 degrees)
        bbox = f"{lon - 0.5},{lat - 0.5},{lon + 0.5},{lat + 0.5}"
        
        # Query for 3 days from start date
        day_range = 3
        
        url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_API_KEY}/{source}/{bbox}/{day_range}/{start_date}"
        
        print(f"  URL: {url}")
        
        try:
            response = requests.get(url, timeout=30)
            
            if response.status_code == 200 and response.text and not response.text.startswith("<!DOCTYPE"):
                # Parse CSV data
                from io import StringIO
                df = pd.read_csv(StringIO(response.text))
                
                if len(df) > 0:
                    print(f"  ✓ Found {len(df)} detections")
                    return df
            
            return None
        
        except Exception as e:
            print(f"  ✗ Error: {e}")
            return None

print("✓ DataCollectorAgent class defined")

In [ ]:
# ============================================================================
# CELL 3: NOISE CLEANER AGENT
# ============================================================================

class NoiseCleanerAgent:
    """
    Agent responsible for cleaning noisy data:
    - Filter Instagram images using GPT-4o-mini vision
    - Filter Twitter text using LLM relevance classification
    """
    
    def __init__(self, data_collector):
        """
        Initialize Noise Cleaner Agent
        
        Args:
            data_collector: DataCollectorAgent instance with collected data
        """
        self.data_collector = data_collector
        self.cleaned_instagram_data = []
        self.cleaned_twitter_data = []
    
    def clean_instagram(self):
        """
        Filter Instagram images using GPT-4o-mini vision analysis
        Keeps only posts where include_in_dataset is True
        
        Returns:
            list: Cleaned Instagram data
        """
        print(f"\n{'='*60}")
        print(f"STEP 4: Cleaning Instagram Data")
        print(f"{'='*60}")
        
        fire_name = self.data_collector.fire_info['fire_name']
        start_date = self.data_collector.fire_info['start_date']
        end_date = self.data_collector.fire_info.get('end_date', 'ongoing')
        
        instagram_data = self.data_collector.instagram_data
        
        print(f"Fire: {fire_name}")
        print(f"Date range: {start_date} to {end_date}")
        print(f"Total Instagram posts to analyze: {len(instagram_data)}")
        print(f"(This may take several minutes)\n")
        
        for idx, metadata in enumerate(instagram_data):
            image_filename = metadata['image_filename']
            image_path = self.data_collector.instagram_images_dir / image_filename
            
            # Check if image exists
            if not image_path.exists():
                print(f"  [{idx+1}/{len(instagram_data)}] ✗ Image not found: {image_filename}")
                continue
            
            # Encode image to base64
            try:
                with open(image_path, 'rb') as f:
                    image_data = base64.b64encode(f.read()).decode('utf-8')
            except Exception as e:
                print(f"  [{idx+1}/{len(instagram_data)}] ✗ Error reading image: {e}")
                continue
            
            # EXACT prompt from your specification
            prompt = f"""You are a noise cleaning agent. Task: Analyze this Instagram image and its metadata to determine if it should be included in the fire event dataset.
**Image Metadata:**
- Upload timestamp: {metadata.get('timestamp', 'N/A')}
- Caption: {metadata.get('caption_clean', 'N/A')}
- Location: {metadata.get('location_name', 'N/A')}
- Instagram URL: {metadata.get('instagram_url', 'N/A')}
**Step 1: Image Content Verification**
Determine if the image shows:
- Real fire, smoke, or smoke plumes
- Taken by a person in the real world (first-person perspective, bystander view, or from a location)
EXCLUDE if the image is:
- A screenshot of news coverage or TV broadcast
- A photo of a screen/monitor
- A map, graphic, or infographic
- Professional news photography (with watermarks/chyrons)
- Artwork, illustration, or digitally created content
- Stock photos or historical images
**Step 2: Temporal Verification**
Check if there are signs the photo was taken on a different date than the upload timestamp:
- Text overlay with a different date
- News ticker/chyron showing a different date
- Caption explicitly mentioning "yesterday," "last week," or a specific past date
- Visual indicators (like "Throwback" or "Archive")
**Respond ONLY with valid JSON in this exact format:**
{{
  "include_in_dataset": true or false,
  "step1_real_fire_smoke": true or false,
  "step1_exclusion_reason": "reason or null",
  "step2_temporal_mismatch": true or false,
  "step2_evidence": "description or null",
  "notes": "brief explanation"
}}"""

            try:
                response = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": prompt
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": f"data:image/jpeg;base64,{image_data}"
                                    }
                                }
                            ]
                        }
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.3,
                    max_tokens=500
                )
                
                result = json.loads(response.choices[0].message.content)

                if result.get('include_in_dataset', False):
                    self.cleaned_instagram_data.append(metadata)
                    print(f"  [{idx+1}/{len(instagram_data)}] ✓ KEPT - {image_filename}")
                else:
                    print(f"  [{idx+1}/{len(instagram_data)}] ✗ FILTERED - {result.get('notes', 'excluded')}")
                
            except Exception as e:
                print(f"  [{idx+1}/{len(instagram_data)}] ✗ Error analyzing: {e}")
                continue
            
            # Rate limiting
            time.sleep(1)
        
        print(f"\n✓ Instagram cleaning complete")
        print(f"  Original: {len(instagram_data)} posts")
        print(f"  Cleaned: {len(self.cleaned_instagram_data)} posts")
        print(f"  Filtered out: {len(instagram_data) - len(self.cleaned_instagram_data)} posts")
        
        return self.cleaned_instagram_data
    
    def clean_twitter(self):
        """
        Filter Twitter data using LLM relevance classification
        Keeps only tweets classified as "RELEVANT" with confidence > 0.6
        
        Returns:
            list: Cleaned Twitter data
        """
        print(f"\n{'='*60}")
        print(f"STEP 5: Cleaning Twitter Data")
        print(f"{'='*60}")
        
        fire_name = self.data_collector.fire_info['fire_name']
        start_date = self.data_collector.fire_info['start_date']
        end_date = self.data_collector.fire_info.get('end_date', 'ongoing')
        
        twitter_data = self.data_collector.twitter_data
        
        print(f"Fire: {fire_name}")
        print(f"Date range: {start_date} to {end_date}")
        print(f"Total tweets to analyze: {len(twitter_data)}")
        print(f"(This may take several minutes)\n")
        
        for idx, tweet in enumerate(twitter_data):
            text = tweet.get('text', '')
            
            if not text:
                continue
            

            prompt = f"""You are a noise cleaning agent. Classify this tweet as RELEVANT or NOT_RELEVANT to the {fire_name} from {start_date} to {end_date}.

RELEVANT = Reporting fire conditions, locations, impacts, evacuations, observations
NOT_RELEVANT = Past fires, jokes, sales, unrelated content

Tweet: "{text}"

Return ONLY valid JSON:
{{"classification": "RELEVANT or NOT_RELEVANT", "confidence": 0.0-1.0, "reason": "brief explanation"}}"""

            try:
                response = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"},
                    temperature=0.3,
                    max_tokens=300
                )
                
                result = json.loads(response.choices[0].message.content)
                

                if result['classification'] == 'RELEVANT' and result.get('confidence', 0) > 0.6:
                    self.cleaned_twitter_data.append(tweet)
                    print(f"  [{idx+1}/{len(twitter_data)}] ✓ KEPT (confidence: {result['confidence']})")
                else:
                    print(f"  [{idx+1}/{len(twitter_data)}] ✗ FILTERED - {result['classification']}")
                
            except Exception as e:
                print(f"  [{idx+1}/{len(twitter_data)}] ✗ Error analyzing: {e}")
                continue
            
            # Rate limiting
            time.sleep(0.5)
        
        print(f"\n✓ Twitter cleaning complete")
        print(f"  Original: {len(twitter_data)} tweets")
        print(f"  Cleaned: {len(self.cleaned_twitter_data)} tweets")
        print(f"  Filtered out: {len(twitter_data) - len(self.cleaned_twitter_data)} tweets")
        
        return self.cleaned_twitter_data

print("✓ NoiseCleanerAgent class defined")

In [ ]:
# ============================================================================
# CELL 4: CLUSTERING AGENT - PART 1 (GEOLOCATION)
# ============================================================================

class ClusteringAgent:
    """
    Agent responsible for:
    1. Geolocating Instagram and Twitter posts
    2. Clustering posts using spatiotemporal DBSCAN
    """
    
    def __init__(self, noise_cleaner):
        """
        Initialize Clustering Agent
        
        Args:
            noise_cleaner: NoiseCleanerAgent instance with cleaned data
        """
        self.noise_cleaner = noise_cleaner
        self.geolocated_instagram_data = []
        self.geolocated_twitter_data = []
        self.clusters = {}
    
    def geolocate_instagram(self):
        """
        Extract location from Instagram captions and images, then geocode
        
        Returns:
            list: Instagram data with location and coordinates filled in
        """
        print(f"\n{'='*60}")
        print(f"STEP 6: Geolocating Instagram Posts")
        print(f"{'='*60}")
        
        fire_name = self.noise_cleaner.data_collector.fire_info['fire_name']
        start_date = self.noise_cleaner.data_collector.fire_info['start_date']
        end_date = self.noise_cleaner.data_collector.fire_info.get('end_date', 'ongoing')
        
        instagram_data = self.noise_cleaner.cleaned_instagram_data
        
        print(f"Fire: {fire_name}")
        print(f"Total Instagram posts to geolocate: {len(instagram_data)}")
        print(f"(This may take several minutes)\n")
        
        for idx, metadata in enumerate(instagram_data):
            caption = metadata.get('caption_clean', '')
            image_filename = metadata['image_filename']
            
            # Step 1: Try to extract location from caption
            location_from_caption = self._extract_location_from_caption(
                caption, fire_name, start_date, end_date
            )
            
            extracted_location = None
            confidence = 0.0
            
            if location_from_caption and location_from_caption.get('confidence', 0) > 0.5:
                extracted_location = location_from_caption['location']
                confidence = location_from_caption['confidence']
                print(f"  [{idx+1}/{len(instagram_data)}] Caption location: {extracted_location} (confidence: {confidence})")
            else:
                # Step 2: If caption fails, try to extract from image
                print(f"  [{idx+1}/{len(instagram_data)}] Caption failed, analyzing image...")
                image_path = self.noise_cleaner.data_collector.instagram_images_dir / image_filename
                
                if image_path.exists():
                    location_from_image = self._extract_location_from_image(
                        image_path, fire_name, start_date, end_date
                    )
                    
                    if location_from_image and location_from_image.get('confidence', 0) > 0.5:
                        extracted_location = location_from_image['location']
                        confidence = location_from_image['confidence']
                        print(f"  [{idx+1}/{len(instagram_data)}] Image location: {extracted_location} (confidence: {confidence})")
            
            # Step 3: Geocode the extracted location
            if extracted_location:
                lat, lon = self._geocode_location(extracted_location)
                
                if lat and lon:
                    metadata['llm_extracted_location'] = extracted_location
                    metadata['coordinates'] = [lat, lon]
                    metadata['llm_confidence'] = confidence
                    self.geolocated_instagram_data.append(metadata)
                    print(f"  [{idx+1}/{len(instagram_data)}] ✓ Geocoded: {lat}, {lon}")
                else:
                    print(f"  [{idx+1}/{len(instagram_data)}] ✗ Geocoding failed for: {extracted_location}")
            else:
                print(f"  [{idx+1}/{len(instagram_data)}] ✗ No location found")
            
            time.sleep(1)  # Rate limiting
        
        print(f"\n✓ Instagram geolocation complete")
        print(f"  Successfully geolocated: {len(self.geolocated_instagram_data)}/{len(instagram_data)}")
        
        return self.geolocated_instagram_data
    
    def _extract_location_from_caption(self, caption, fire_name, start_date, end_date):
        """Extract location from Instagram caption using LLM"""
        if not caption:
            return None
        
        prompt = f"""You are a clustering agent. Extract the most specific geographic location from this Instagram caption related to the {fire_name} wildfire from {start_date} to {end_date}.

Caption: "{caption}"

Extract the MOST SPECIFIC location possible:
- Specific landmarks, intersections, addresses (e.g., "Highway 99 and Skyway exit")
- Neighborhoods, parks, specific areas (e.g., "Bidwell Park north entrance")
- Streets, buildings, recognizable places
- Include city/county/state for context

If NO location is mentioned, return null.

Return ONLY valid JSON:
{{
  "location": "most specific location string or null",
  "confidence": 0.0-1.0
}}"""

        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.3,
                max_tokens=200
            )
            
            result = json.loads(response.choices[0].message.content)
            return result
        
        except Exception as e:
            print(f"    Error extracting location from caption: {e}")
            return None
    
    def _extract_location_from_image(self, image_path, fire_name, start_date, end_date):
        """Extract location from Instagram image using GPT-4o vision"""
        try:
            # Encode image to base64
            with open(image_path, 'rb') as f:
                image_data = base64.b64encode(f.read()).decode('utf-8')
            
            prompt = f"""You are a clustering agent. Analyze this image related to the {fire_name} wildfire from {start_date} to {end_date} to extract geographic location clues.

Look for:
- Road signs, highway markers, mile markers
- Business names, store signs
- Landmarks, buildings, geographical features
- Street signs, intersection names
- Any text visible in the image indicating location

Extract the MOST SPECIFIC location possible based on visual clues.

Return ONLY valid JSON:
{{
  "location": "most specific location string or null",
  "confidence": 0.0-1.0,
  "visual_clues": "what you see that indicates location"
}}"""

            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": prompt
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{image_data}"
                                }
                            }
                        ]
                    }
                ],
                response_format={"type": "json_object"},
                temperature=0.3,
                max_tokens=300
            )
            
            result = json.loads(response.choices[0].message.content)
            return result
        
        except Exception as e:
            print(f"    Error extracting location from image: {e}")
            return None
    
    def geolocate_twitter(self):
        """
        Extract location from Twitter text, then geocode
        
        Returns:
            list: Twitter data with location and coordinates added
        """
        print(f"\n{'='*60}")
        print(f"STEP 7: Geolocating Twitter Posts")
        print(f"{'='*60}")
        
        fire_name = self.noise_cleaner.data_collector.fire_info['fire_name']
        start_date = self.noise_cleaner.data_collector.fire_info['start_date']
        end_date = self.noise_cleaner.data_collector.fire_info.get('end_date', 'ongoing')
        
        twitter_data = self.noise_cleaner.cleaned_twitter_data
        
        print(f"Fire: {fire_name}")
        print(f"Total tweets to geolocate: {len(twitter_data)}")
        print(f"(This may take several minutes)\n")
        
        for idx, tweet in enumerate(twitter_data):
            text = tweet.get('text', '')
            
            if not text:
                continue
            
            # Extract location from tweet text
            prompt = f"""You are a clustering agent. Extract the most specific geographic location from this tweet related to the {fire_name} wildfire from {start_date} to {end_date}.

Tweet: "{text}"

Extract the MOST SPECIFIC location possible:
- Specific landmarks, intersections, addresses
- Neighborhoods, parks, specific areas
- Streets, buildings, recognizable places
- Include city/county/state for context

If NO location is mentioned, return null.

Return ONLY valid JSON:
{{
  "location": "most specific location string or null",
  "confidence": 0.0-1.0
}}"""

            try:
                response = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"},
                    temperature=0.3,
                    max_tokens=200
                )
                
                result = json.loads(response.choices[0].message.content)
                
                if result.get('location') and result.get('confidence', 0) > 0.5:
                    extracted_location = result['location']
                    confidence = result['confidence']
                    
                    # Geocode the location
                    lat, lon = self._geocode_location(extracted_location)
                    
                    if lat and lon:
                        tweet['extracted_location'] = extracted_location
                        tweet['coordinates'] = [lat, lon]
                        tweet['location_confidence'] = confidence
                        self.geolocated_twitter_data.append(tweet)
                        print(f"  [{idx+1}/{len(twitter_data)}] ✓ {extracted_location} → {lat}, {lon}")
                    else:
                        print(f"  [{idx+1}/{len(twitter_data)}] ✗ Geocoding failed for: {extracted_location}")
                else:
                    print(f"  [{idx+1}/{len(twitter_data)}] ✗ No location found")
                
            except Exception as e:
                print(f"  [{idx+1}/{len(twitter_data)}] ✗ Error: {e}")
                continue
            
            time.sleep(0.5)  # Rate limiting
        
        print(f"\n✓ Twitter geolocation complete")
        print(f"  Successfully geolocated: {len(self.geolocated_twitter_data)}/{len(twitter_data)}")
        
        return self.geolocated_twitter_data
    
    def _geocode_location(self, location_name):
        """
        Geocode a location name to lat/lon using Nominatim
        
        Args:
            location_name: Location string to geocode
        
        Returns:
            tuple: (latitude, longitude) or (None, None) if failed
        """
        try:
            time.sleep(1)  # Rate limiting for Nominatim
            location = geolocator.geocode(location_name)
            
            if location:
                return location.latitude, location.longitude
            else:
                return None, None
        
        except GeocoderTimedOut:
            print(f"    Geocoding timed out")
            return None, None
        except Exception as e:
            print(f"    Geocoding error: {e}")
            return None, None

print("✓ ClusteringAgent class defined (Part 1: Geolocation methods)")

In [ ]:
# ============================================================================
# CELL 5: FIRE ANALYST AGENT
# ============================================================================

class FireAnalystAgent:
    """
    Agent responsible for:
    1. Collecting meteorological data for each cluster
    2. Analyzing clusters: summarizing social media + FIRMS + weather
    3. Generating risk assessment reports
    """
    
    def __init__(self, clustering_agent):
        """
        Initialize Fire Analyst Agent
        
        Args:
            clustering_agent: ClusteringAgent instance with clustered data
        """
        self.clustering_agent = clustering_agent
        self.cluster_reports = {}
    
    def analyze_all_clusters(self):
        """
        Analyze all clusters and generate reports
        
        Returns:
            dict: Cluster reports (JSON + HTML for each cluster)
        """
        print(f"\n{'='*60}")
        print(f"STEP 9: Analyzing Clusters and Generating Reports")
        print(f"{'='*60}")
        
        clusters = self.clustering_agent.clusters
        firms_data = self.clustering_agent.noise_cleaner.data_collector.firms_data
        fire_info = self.clustering_agent.noise_cleaner.data_collector.fire_info
        
        print(f"Total clusters to analyze: {len(clusters)}")
        
        if firms_data is None or len(firms_data) == 0:
            print("\n⚠️ Warning: No FIRMS data available")
        
        for cluster_id, cluster_data in clusters.items():
            print(f"\n{'='*60}")
            print(f"Analyzing {cluster_id}")
            print(f"{'='*60}")
            
            # Step 1: Summarize social media 
            smd_perspective = self._generate_smd_perspective(cluster_data, fire_info)
            
            # Step 2: Get meteorological data for cluster start time
            weather_data = self._get_cluster_weather(cluster_data)
            
            # Step 3: Find nearest FIRMS pixel (date-matched)
            firms_analysis = self._analyze_firms_proximity(cluster_data, firms_data)
            
            # Step 4: Generate risk assessment report
            report = self._generate_risk_assessment(
                cluster_id,
                cluster_data,
                smd_perspective,
                weather_data,
                firms_analysis,
                fire_info
            )
            
            # Store reports
            self.cluster_reports[cluster_id] = report
            
            print(f"✓ {cluster_id} report generated")
            print(f"  Risk Level: {report['json']['risk_level']}")
        
        print(f"\n{'='*60}")
        print(f"✓ All cluster reports generated: {len(self.cluster_reports)}")
        print(f"{'='*60}")
        
        return self.cluster_reports
    
    def _generate_smd_perspective(self, cluster_data, fire_info):
        """
        Generate Social Media Data (SMD) perspective by summarizing all posts
        
        Args:
            cluster_data: Single cluster dict with Instagram and Twitter posts
            fire_info: Fire information
        
        Returns:
            dict: SMD perspective with Instagram summary, Twitter summary, combined
        """
        instagram_posts = cluster_data['instagram_posts']
        twitter_posts = cluster_data['twitter_posts']
        
        fire_name = fire_info['fire_name']
        
        print(f"  Generating SMD Perspective...")
        print(f"    Instagram posts: {len(instagram_posts)}")
        print(f"    Twitter posts: {len(twitter_posts)}")
        
        # Summarize Instagram captions
        instagram_summary = ""
        if len(instagram_posts) > 0:
            captions = [post.get('caption_clean', '') for post in instagram_posts]
            captions_text = "\n".join([f"- {cap}" for cap in captions if cap])
            
            prompt = f"""You are a fire analyst. Summarize these Instagram captions about the {fire_name} wildfire.

Instagram Captions:
{captions_text}

Create a brief summary (2-3 sentences) of what people observed and reported. Focus on:
- Fire/smoke observations
- Locations mentioned
- Timing and progression
- Impacts and concerns

Return ONLY the summary text, no JSON."""

            try:
                response = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3,
                    max_tokens=300
                )
                instagram_summary = response.choices[0].message.content.strip()
                print(f"    ✓ Instagram summary generated")
            except Exception as e:
                print(f"    ✗ Error generating Instagram summary: {e}")
                instagram_summary = f"Based on {len(instagram_posts)} Instagram posts with fire observations."
        
        # Summarize Twitter posts
        twitter_summary = ""
        if len(twitter_posts) > 0:
            tweets = [post.get('text', '') for post in twitter_posts]
            tweets_text = "\n".join([f"- {tweet}" for tweet in tweets if tweet])
            
            prompt = f"""You are a fire analyst. Summarize these tweets about the {fire_name} wildfire.

Tweets:
{tweets_text}

Create a brief summary (2-3 sentences) of what people observed and reported. Focus on:
- Fire/smoke observations
- Locations mentioned
- Timing and progression
- Impacts and concerns

Return ONLY the summary text, no JSON."""

            try:
                response = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3,
                    max_tokens=300
                )
                twitter_summary = response.choices[0].message.content.strip()
                print(f"    ✓ Twitter summary generated")
            except Exception as e:
                print(f"    ✗ Error generating Twitter summary: {e}")
                twitter_summary = f"Based on {len(twitter_posts)} tweets with fire reports."
        
        # Combine summaries
        combined_summary = ""
        if instagram_summary and twitter_summary:
            combined_summary = f"Instagram: {instagram_summary}\n\nTwitter: {twitter_summary}"
        elif instagram_summary:
            combined_summary = instagram_summary
        elif twitter_summary:
            combined_summary = twitter_summary
        else:
            combined_summary = "No social media observations available."
        
        return {
            'instagram_summary': instagram_summary,
            'twitter_summary': twitter_summary,
            'combined_summary': combined_summary
        }
    
    def _get_cluster_weather(self, cluster_data):
        """
        Get meteorological data for cluster (at start time)
        
        Args:
            cluster_data: Single cluster dict
        
        Returns:
            dict: Weather data or None
        """
        centroid = cluster_data['centroid']
        time_range = cluster_data['time_range']
        
        # Parse start time
        try:
            start_dt = pd.to_datetime(time_range['start'])
            date_str = start_dt.strftime('%Y-%m-%d')
            hour = start_dt.hour
            
            print(f"  Fetching weather data for {date_str} at hour {hour}...")
            
            weather = self._get_weather(
                centroid['lat'],
                centroid['lon'],
                date_str,
                hour
            )
            
            return weather
        
        except Exception as e:
            print(f"  ✗ Error getting weather: {e}")
            return None
    
    def _get_weather(self, lat, lon, date, hour=12):
        """
        Get historical weather data from Open-Meteo Archive API
        (From Reference Cell 6)
        
        Args:
            lat, lon: Location coordinates
            date: Date string (YYYY-MM-DD)
            hour: Hour of day (0-23)
        
        Returns:
            dict: Weather data or None
        """
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            'latitude': lat,
            'longitude': lon,
            'start_date': date,
            'end_date': date,
            'hourly': 'wind_speed_10m,wind_direction_10m,wind_gusts_10m,temperature_2m,relative_humidity_2m'
        }
        
        try:
            response = requests.get(url, params=params, timeout=30)
            data = response.json()
            
            hourly = data['hourly']
            
            # Convert wind direction to compass bearing
            def deg_to_compass(deg):
                dirs = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
                ix = round(deg / 45) % 8
                return dirs[ix]
            
            weather_data = {
                'speed_mph': hourly['wind_speed_10m'][hour] * 0.621371,
                'direction_deg': hourly['wind_direction_10m'][hour],
                'direction_text': deg_to_compass(hourly['wind_direction_10m'][hour]),
                'gusts_mph': hourly['wind_gusts_10m'][hour] * 0.621371,
                'temp_f': hourly['temperature_2m'][hour] * 9/5 + 32,
                'humidity_pct': hourly['relative_humidity_2m'][hour]
            }
            
            print(f"    ✓ Weather: {weather_data['speed_mph']:.1f} mph {weather_data['direction_text']}, {weather_data['humidity_pct']:.0f}% humidity")
            return weather_data
        
        except Exception as e:
            print(f"    ⚠️ Weather query failed: {e}")
            return None
    
    def _analyze_firms_proximity(self, cluster_data, firms_data):
        """
        Find nearest FIRMS fire pixel (date-matched) and calculate distance
        
        Args:
            cluster_data: Single cluster dict
            firms_data: DataFrame with FIRMS detections
        
        Returns:
            dict: FIRMS analysis with nearest pixel info
        """
        if firms_data is None or len(firms_data) == 0:
            return {
                'nearest_distance_km': None,
                'nearest_pixel': None,
                'temporal_comparison': "No satellite data available"
            }
        
        centroid = cluster_data['centroid']
        time_range = cluster_data['time_range']
        
        # Get cluster date
        try:
            cluster_start = pd.to_datetime(time_range['start'])
            cluster_date = cluster_start.strftime('%Y-%m-%d')
        except:
            cluster_date = None
        
        # Filter FIRMS by date if available
        if cluster_date and 'acq_date' in firms_data.columns:
            date_matched_firms = firms_data[firms_data['acq_date'] == cluster_date]
            
            if len(date_matched_firms) == 0:
                print(f"    ⚠️ No FIRMS detections on {cluster_date}")
                date_matched_firms = firms_data  # Fall back to all data
        else:
            date_matched_firms = firms_data
        
        # Calculate distances to all FIRMS pixels
        print(f"    Calculating distances to {len(date_matched_firms)} FIRMS pixels...")
        
        min_distance = float('inf')
        nearest_pixel = None
        
        for idx, row in date_matched_firms.iterrows():
            firms_lat = row.get('latitude', row.get('LATITUDE'))
            firms_lon = row.get('longitude', row.get('LONGITUDE'))
            
            if pd.isna(firms_lat) or pd.isna(firms_lon):
                continue
            
            distance = self._haversine_distance(
                centroid['lat'], centroid['lon'],
                firms_lat, firms_lon
            )
            
            if distance < min_distance:
                min_distance = distance
                nearest_pixel = row
        
        if nearest_pixel is not None:
            # Extract acquisition time
            acq_date = nearest_pixel.get('acq_date', nearest_pixel.get('ACQ_DATE', 'Unknown'))
            acq_time = nearest_pixel.get('acq_time', nearest_pixel.get('ACQ_TIME', 'Unknown'))
            
            if acq_time != 'Unknown' and len(str(acq_time)) == 4:
                acq_time_str = str(acq_time).zfill(4)
                acq_time_formatted = f"{acq_time_str[:2]}:{acq_time_str[2:]} UTC"
            else:
                acq_time_formatted = str(acq_time)
            
            # Temporal comparison
            try:
                cluster_start_dt = pd.to_datetime(time_range['start'])
                
                # Parse FIRMS datetime
                firms_time_str = f"{acq_date} {str(acq_time).zfill(4)}"
                firms_dt = pd.to_datetime(firms_time_str, format='%Y-%m-%d %H%M')
                
                time_diff = (cluster_start_dt - firms_dt).total_seconds() / 3600  # Hours
                
                if time_diff > 0:
                    temporal_comparison = f"Social media reports started {abs(time_diff):.1f} hours AFTER satellite detection"
                else:
                    temporal_comparison = f"Social media reports started {abs(time_diff):.1f} hours BEFORE satellite detection"
            except:
                temporal_comparison = "Temporal comparison unavailable"
            
            print(f"    ✓ Nearest FIRMS pixel: {min_distance:.2f} km away")
            print(f"      Detected: {acq_date} at {acq_time_formatted}")
            
            return {
                'nearest_distance_km': round(min_distance, 2),
                'nearest_pixel': {
                    'acq_date': acq_date,
                    'acq_time': acq_time_formatted,
                    'latitude': float(nearest_pixel.get('latitude', nearest_pixel.get('LATITUDE'))),
                    'longitude': float(nearest_pixel.get('longitude', nearest_pixel.get('LONGITUDE'))),
                    'brightness': float(nearest_pixel.get('brightness', nearest_pixel.get('BRIGHTNESS', 0))),
                    'frp': float(nearest_pixel.get('frp', nearest_pixel.get('FRP', 0)))
                },
                'temporal_comparison': temporal_comparison
            }
        else:
            print(f"    ✗ No FIRMS pixels found")
            return {
                'nearest_distance_km': None,
                'nearest_pixel': None,
                'temporal_comparison': "No satellite detections found"
            }
    
    def _haversine_distance(self, lat1, lon1, lat2, lon2):
        """Calculate distance between two points in kilometers"""
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        r = 6371
        return c * r
    
    def _generate_risk_assessment(self, cluster_id, cluster_data, smd_perspective, weather_data, firms_analysis, fire_info):
        """
        Generate final risk assessment report using LLM
        
        Args:
            cluster_id: Cluster identifier
            cluster_data: Cluster dict
            smd_perspective: SMD summaries
            weather_data: Weather dict
            firms_analysis: FIRMS proximity analysis
            fire_info: Fire information
        
        Returns:
            dict: Report with JSON and HTML versions
        """
        print(f"  Generating risk assessment...")
        
        fire_name = fire_info['fire_name']
        
        # Prepare weather string
        if weather_data:
            weather_str = f"""Wind: {weather_data['speed_mph']:.1f} mph {weather_data['direction_text']} (gusts: {weather_data['gusts_mph']:.1f} mph)
Temperature: {weather_data['temp_f']:.1f}°F
Humidity: {weather_data['humidity_pct']:.0f}%"""
        else:
            weather_str = "Weather data unavailable"
        
        # Prepare FIRMS string
        if firms_analysis['nearest_distance_km']:
            firms_str = f"""Nearest satellite fire detection: {firms_analysis['nearest_distance_km']} km away
Detected: {firms_analysis['nearest_pixel']['acq_date']} at {firms_analysis['nearest_pixel']['acq_time']}
{firms_analysis['temporal_comparison']}"""
        else:
            firms_str = "No satellite fire detections available for comparison"
        
        # LLM prompt for risk assessment
        prompt = f"""You are a fire analyst. Based on the following information about the {fire_name} wildfire, assess the risk level.

**Social Media Reports Summary:**
{smd_perspective['combined_summary']}

**Satellite Fire Detection:**
{firms_str}

**Meteorological Conditions:**
{weather_str}

**Cluster Details:**
- Time Range: {cluster_data['time_range']['start']} to {cluster_data['time_range']['end']}
- Location: {cluster_data['centroid']['lat']:.6f}, {cluster_data['centroid']['lon']:.6f}
- Total Posts: {cluster_data['total_posts']} (Instagram: {len(cluster_data['instagram_posts'])}, Twitter: {len(cluster_data['twitter_posts'])})

Provide a risk assessment with:
1. Risk Level (Low/Moderate/High/Extreme)
2. Reasoning based on proximity to fire, weather conditions, and social media reports
3. Key concerns

Return ONLY valid JSON:
{{
  "risk_level": "Low/Moderate/High/Extreme",
  "reasoning": "detailed reasoning based on all factors",
  "key_concerns": ["concern1", "concern2", "concern3"]
}}"""

        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.3,
                max_tokens=500
            )
            
            risk_assessment = json.loads(response.choices[0].message.content)
            print(f"    ✓ Risk assessment: {risk_assessment['risk_level']}")
        
        except Exception as e:
            print(f"    ✗ Error generating risk assessment: {e}")
            risk_assessment = {
                "risk_level": "Unknown",
                "reasoning": "Error generating assessment",
                "key_concerns": []
            }
        
        # Create JSON report
        json_report = {
            'cluster_id': cluster_id,
            'risk_level': risk_assessment['risk_level'],
            'reasoning': risk_assessment['reasoning'],
            'key_concerns': risk_assessment['key_concerns'],
            'smd_perspective': smd_perspective,
            'firms_analysis': firms_analysis,
            'weather': weather_data,
            'cluster_info': {
                'time_range': cluster_data['time_range'],
                'centroid': cluster_data['centroid'],
                'total_posts': cluster_data['total_posts'],
                'instagram_count': len(cluster_data['instagram_posts']),
                'twitter_count': len(cluster_data['twitter_posts'])
            }
        }
        
        
        html_report = self._format_html_report(json_report)
        
        return {
            'json': json_report,
            'html': html_report
        }
    
    def _format_html_report(self, json_report):
        """
        Format report as HTML for Folium popup
        
        Args:
            json_report: Report dict
        
        Returns:
            str: HTML formatted report
        """
        risk_level = json_report['risk_level']
        
        # Color-code risk level
        risk_colors = {
            'Low': 'green',
            'Moderate': 'orange',
            'High': 'red',
            'Extreme': 'darkred',
            'Unknown': 'gray'
        }
        risk_color = risk_colors.get(risk_level, 'gray')
        
        html = f"""
        <div style="width: 400px; max-height: 500px; overflow-y: auto; font-family: Arial, sans-serif;">
            <h3 style="margin-top: 0; color: {risk_color};">Risk Level: {risk_level}</h3>
            
            <p><strong>Time Range:</strong><br/>
            {json_report['cluster_info']['time_range']['start']} to<br/>
            {json_report['cluster_info']['time_range']['end']}</p>
            
            <p><strong>Posts:</strong> {json_report['cluster_info']['total_posts']} total 
            ({json_report['cluster_info']['instagram_count']} Instagram, 
            {json_report['cluster_info']['twitter_count']} Twitter)</p>
            
            <hr/>
            
            <p><strong>Social Media Observations:</strong><br/>
            {json_report['smd_perspective']['combined_summary'].replace(chr(10), '<br/>')}</p>
            
            <hr/>
            
            <p><strong>Satellite Detection:</strong><br/>
        """
        
        if json_report['firms_analysis']['nearest_distance_km']:
            html += f"""
            Distance: {json_report['firms_analysis']['nearest_distance_km']} km<br/>
            Detected: {json_report['firms_analysis']['nearest_pixel']['acq_date']} 
            at {json_report['firms_analysis']['nearest_pixel']['acq_time']}<br/>
            <em>{json_report['firms_analysis']['temporal_comparison']}</em>
            """
        else:
            html += "No satellite detections available"
        
        html += "</p><hr/><p><strong>Weather Conditions:</strong><br/>"
        
        if json_report['weather']:
            w = json_report['weather']
            html += f"""
            Wind: {w['speed_mph']:.1f} mph {w['direction_text']} (gusts: {w['gusts_mph']:.1f} mph)<br/>
            Temperature: {w['temp_f']:.1f}°F<br/>
            Humidity: {w['humidity_pct']:.0f}%
            """
        else:
            html += "Weather data unavailable"
        
        html += f"""
            </p>
            
            <hr/>
            
            <p><strong>Risk Assessment:</strong><br/>
            {json_report['reasoning']}</p>
            
            <p><strong>Key Concerns:</strong></p>
            <ul>
        """
        
        for concern in json_report['key_concerns']:
            html += f"<li>{concern}</li>"
        
        html += """
            </ul>
        </div>
        """
        
        return html

print("✓ FireAnalystAgent class defined")

In [ ]:
# ============================================================================
# CELL 6: VISUALIZATION AGENT
# ============================================================================

class VisualizationAgent:
    """
    Agent responsible for creating interactive Folium map with:
    - Layer control for Day 1, Day 2, Last Day
    - Cluster markers with report popups
    - FIRMS fire pixels
    """
    
    def __init__(self, fire_analyst):
        """
        Initialize Visualization Agent
        
        Args:
            fire_analyst: FireAnalystAgent instance with cluster reports
        """
        self.fire_analyst = fire_analyst
        self.map_file = None
    
    def create_interactive_map(self):
        """
        Create interactive Folium map with day-based layer control
        
        Returns:
            str: Path to saved HTML map file
        """
        print(f"\n{'='*60}")
        print(f"STEP 10: Creating Interactive Visualization")
        print(f"{'='*60}")
        
        clusters = self.fire_analyst.clustering_agent.clusters
        cluster_reports = self.fire_analyst.cluster_reports
        firms_data = self.fire_analyst.clustering_agent.noise_cleaner.data_collector.firms_data
        fire_info = self.fire_analyst.clustering_agent.noise_cleaner.data_collector.fire_info
        
        print(f"Fire: {fire_info['fire_name']}")
        print(f"Clusters: {len(clusters)}")
        print(f"FIRMS detections: {len(firms_data) if firms_data is not None else 0}")
        
        # Determine days to visualize
        start_date = fire_info['start_date']
        day_1 = pd.to_datetime(start_date)
        day_2 = day_1 + timedelta(days=1)
        
        # Find last day from cluster data
        all_cluster_dates = []
        for cluster_data in clusters.values():
            try:
                cluster_end = pd.to_datetime(cluster_data['time_range']['end'])
                all_cluster_dates.append(cluster_end)
            except:
                pass
        
        if all_cluster_dates:
            last_day = max(all_cluster_dates)
        else:
            last_day = day_1 + timedelta(days=2)
        
        day_1_str = day_1.strftime('%Y-%m-%d')
        day_2_str = day_2.strftime('%Y-%m-%d')
        last_day_str = last_day.strftime('%Y-%m-%d')
        
        print(f"\nVisualization days:")
        print(f"  Day 1: {day_1_str}")
        print(f"  Day 2: {day_2_str}")
        print(f"  Last Day: {last_day_str}")
        
        # Calculate map center from all clusters
        all_lats = [c['centroid']['lat'] for c in clusters.values()]
        all_lons = [c['centroid']['lon'] for c in clusters.values()]
        
        if all_lats:
            center_lat = np.mean(all_lats)
            center_lon = np.mean(all_lons)
        else:
            center_lat = fire_info['lat']
            center_lon = fire_info['lon']
        
        print(f"\nMap center: {center_lat:.6f}, {center_lon:.6f}")
        
        # Create base map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=11,
            tiles='OpenStreetMap'
        )
        
        # Fixed color palette for clusters
        colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
                  'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
                  'darkpurple', 'pink', 'lightblue', 'lightgreen', 'gray',
                  'black', 'lightgray']
        
        # Assign colors to clusters
        cluster_colors = {}
        for idx, cluster_id in enumerate(sorted(clusters.keys())):
            cluster_colors[cluster_id] = colors[idx % len(colors)]
        
        # Create feature groups for each day
        day1_group = folium.FeatureGroup(name=f'Day 1: {day_1_str}', show=True)
        day2_group = folium.FeatureGroup(name=f'Day 2: {day_2_str}', show=False)
        last_day_group = folium.FeatureGroup(name=f'Last Day: {last_day_str}', show=False)
        
        # Add FIRMS and clusters to appropriate day groups
        self._add_firms_to_day(firms_data, day_1_str, day1_group)
        self._add_firms_to_day(firms_data, day_2_str, day2_group)
        self._add_firms_to_day(firms_data, last_day_str, last_day_group)
        
        self._add_clusters_to_day(clusters, cluster_reports, cluster_colors, day_1_str, day1_group)
        self._add_clusters_to_day(clusters, cluster_reports, cluster_colors, day_2_str, day2_group)
        self._add_clusters_to_day(clusters, cluster_reports, cluster_colors, last_day_str, last_day_group)
        
        # Add feature groups to map
        day1_group.add_to(m)
        day2_group.add_to(m)
        last_day_group.add_to(m)
        
        # Add layer control
        folium.LayerControl(collapsed=False).add_to(m)
        
        # Add title
        title_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 400px; height: 60px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:16px; padding: 10px">
            <b>{fire_info['fire_name']} Wildfire Analysis</b><br>
            Social Media + Satellite Detection
        </div>
        '''
        m.get_root().html.add_child(folium.Element(title_html))
        
        # Save map
        self.map_file = 'wildfire_analysis_map.html'
        m.save(self.map_file)
        
        print(f"\n✓ Interactive map created: {self.map_file}")
        
        return self.map_file
    
    def _add_firms_to_day(self, firms_data, date_str, feature_group):
        """
        Add FIRMS fire pixels for specific date to feature group
        
        Args:
            firms_data: DataFrame with FIRMS detections
            date_str: Date string (YYYY-MM-DD)
            feature_group: Folium FeatureGroup to add markers to
        """
        if firms_data is None or len(firms_data) == 0:
            return
        
        # Filter by date
        if 'acq_date' in firms_data.columns:
            day_firms = firms_data[firms_data['acq_date'] == date_str]
        elif 'ACQ_DATE' in firms_data.columns:
            day_firms = firms_data[firms_data['ACQ_DATE'] == date_str]
        else:
            day_firms = firms_data
        
        print(f"  Adding {len(day_firms)} FIRMS pixels for {date_str}")
        
        for idx, row in day_firms.iterrows():
            lat = row.get('latitude', row.get('LATITUDE'))
            lon = row.get('longitude', row.get('LONGITUDE'))
            brightness = row.get('brightness', row.get('BRIGHTNESS', 'N/A'))
            acq_time = row.get('acq_time', row.get('ACQ_TIME', 'N/A'))
            
            if pd.isna(lat) or pd.isna(lon):
                continue
            
            # Format time
            if acq_time != 'N/A' and len(str(acq_time)) == 4:
                time_str = str(acq_time).zfill(4)
                acq_time_formatted = f"{time_str[:2]}:{time_str[2:]}"
            else:
                acq_time_formatted = str(acq_time)
            
            popup_text = f"""
            <b>Satellite Fire Detection</b><br>
            Time: {acq_time_formatted} UTC<br>
            Brightness: {brightness}K
            """
            
            folium.CircleMarker(
                location=[lat, lon],
                radius=4,
                color='red',
                fill=True,
                fillColor='red',
                fillOpacity=0.6,
                popup=folium.Popup(popup_text, max_width=200)
            ).add_to(feature_group)
    
    def _add_clusters_to_day(self, clusters, cluster_reports, cluster_colors, date_str, feature_group):
        """
        Add cluster markers for specific date to feature group
        
        Args:
            clusters: Dict of clusters
            cluster_reports: Dict of cluster reports
            cluster_colors: Dict mapping cluster_id to color
            date_str: Date string (YYYY-MM-DD)
            feature_group: Folium FeatureGroup to add markers to
        """
        clusters_added = 0
        
        for cluster_id, cluster_data in clusters.items():
            # Check if cluster has posts on this date
            cluster_start = pd.to_datetime(cluster_data['time_range']['start'])
            cluster_end = pd.to_datetime(cluster_data['time_range']['end'])
            target_date = pd.to_datetime(date_str)
            
            # Check if cluster time range overlaps with target date
            if not (cluster_start.date() <= target_date.date() <= cluster_end.date()):
                continue
            
            clusters_added += 1
            
            centroid = cluster_data['centroid']
            color = cluster_colors[cluster_id]
            
            # Get report HTML
            if cluster_id in cluster_reports:
                popup_html = cluster_reports[cluster_id]['html']
            else:
                popup_html = f"<b>{cluster_id}</b><br>No report available"
            
            # Create marker
            folium.Marker(
                location=[centroid['lat'], centroid['lon']],
                popup=folium.Popup(popup_html, max_width=450),
                icon=folium.Icon(color=color, icon='fire', prefix='fa'),
                tooltip=f"{cluster_id}: {cluster_data['total_posts']} posts"
            ).add_to(feature_group)
        
        print(f"  Adding {clusters_added} clusters for {date_str}")

print("✓ VisualizationAgent class defined")

In [ ]:
# ============================================================================
# CELL 7: MAIN EXECUTION PIPELINE + CLEANUP
# ============================================================================

def cleanup_raw_data(data_collector):
    """
    Delete all raw data for privacy
    Keeps only: map HTML + cluster reports JSON
    
    Args:
        data_collector: DataCollectorAgent instance
    """
    print(f"\n{'='*60}")
    print(f"CLEANUP: Deleting Raw Data")
    print(f"{'='*60}")
    
    import shutil
    
    # Delete Instagram images
    if data_collector.instagram_images_dir.exists():
        print(f"Deleting Instagram images...")
        shutil.rmtree(data_collector.instagram_images_dir)
        print(f"  ✓ Deleted: {data_collector.instagram_images_dir}")
    
    # Delete Instagram metadata
    if data_collector.instagram_metadata_dir.exists():
        print(f"Deleting Instagram metadata...")
        shutil.rmtree(data_collector.instagram_metadata_dir)
        print(f"  ✓ Deleted: {data_collector.instagram_metadata_dir}")
    
    # Delete Twitter directory
    if data_collector.twitter_dir.exists():
        print(f"Deleting Twitter data...")
        shutil.rmtree(data_collector.twitter_dir)
        print(f"  ✓ Deleted: {data_collector.twitter_dir}")
    
    # Delete FIRMS directory
    if data_collector.firms_dir.exists():
        print(f"Deleting FIRMS data...")
        shutil.rmtree(data_collector.firms_dir)
        print(f"  ✓ Deleted: {data_collector.firms_dir}")
    
    # Delete base raw_data directory if empty
    if data_collector.base_dir.exists():
        try:
            data_collector.base_dir.rmdir()
            print(f"  ✓ Deleted: {data_collector.base_dir}")
        except:
            pass  # Directory not empty, leave it
    
    print(f"\n✓ Cleanup complete - all raw data deleted")
    print(f"✓ Preserved files:")
    print(f"  - wildfire_analysis_map.html")
    print(f"  - cluster_reports.json")

def save_cluster_reports(fire_analyst, filename='cluster_reports.json'):
    """
    Save all cluster reports to JSON file
    
    Args:
        fire_analyst: FireAnalystAgent instance
        filename: Output filename
    """
    print(f"\nSaving cluster reports to {filename}...")
    
    # Extract JSON reports only (not HTML)
    reports_to_save = {}
    for cluster_id, report in fire_analyst.cluster_reports.items():
        reports_to_save[cluster_id] = report['json']
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(reports_to_save, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Cluster reports saved: {filename}")

def run_wildfire_analysis_pipeline():
    """
    Main execution pipeline for wildfire analysis
    Runs all agents sequentially and handles errors gracefully
    """
    print("="*80)
    print("WILDFIRE ANALYSIS PIPELINE")
    print("Multi-Agent System: Social Media + Satellite + Weather")
    print("="*80)
    
    try:
        # Get user input
        user_input = input("\nEnter fire name: ")
        
        if not user_input.strip():
            print("Error: Fire name cannot be empty")
            return
        
        print(f"\nStarting analysis for: {user_input}")
        print("="*80)
        
        # STEP 1: Data Collection
        print("\n[AGENT 1/5] DATA COLLECTOR")
        data_collector = DataCollectorAgent()
        
        fire_info = data_collector.extract_fire_info(user_input)
        
        if not fire_info:
            print("\n✗ Error: Could not extract fire information")
            return
        
        # Collect social media
        instagram_data, twitter_data = data_collector.collect_social_media()
        
        if len(instagram_data) == 0 and len(twitter_data) == 0:
            print("\n⚠️ Warning: No social media data collected")
            print("This is expected with APIFY_API_KEY = 'example'")
            print("Pipeline will continue with empty data...")
        
        # Collect FIRMS
        firms_data = data_collector.collect_firms()
        
        if firms_data is None or len(firms_data) == 0:
            print("\n⚠️ Warning: No FIRMS satellite data found")
            print("Pipeline will continue without satellite data...")
        
        # STEP 2: Noise Cleaning
        print("\n[AGENT 2/5] NOISE CLEANER")
        noise_cleaner = NoiseCleanerAgent(data_collector)
        
        if len(instagram_data) > 0:
            cleaned_instagram = noise_cleaner.clean_instagram()
        else:
            print("Skipping Instagram cleaning (no data)")
            cleaned_instagram = []
        
        if len(twitter_data) > 0:
            cleaned_twitter = noise_cleaner.clean_twitter()
        else:
            print("Skipping Twitter cleaning (no data)")
            cleaned_twitter = []
        
        if len(cleaned_instagram) == 0 and len(cleaned_twitter) == 0:
            print("\n⚠️ Warning: No data remained after cleaning")
            print("Cannot proceed with clustering")
            return
        
        # STEP 3: Clustering (Geolocation + DBSCAN)
        print("\n[AGENT 3/5] CLUSTERING")
        clustering_agent = ClusteringAgent(noise_cleaner)
        
        # Geolocate
        if len(cleaned_instagram) > 0:
            geolocated_instagram = clustering_agent.geolocate_instagram()
        else:
            print("Skipping Instagram geolocation (no data)")
            geolocated_instagram = []
        
        if len(cleaned_twitter) > 0:
            geolocated_twitter = clustering_agent.geolocate_twitter()
        else:
            print("Skipping Twitter geolocation (no data)")
            geolocated_twitter = []
        
        if len(geolocated_instagram) == 0 and len(geolocated_twitter) == 0:
            print("\n⚠️ Warning: No posts could be geolocated")
            print("Cannot proceed with clustering")
            return
        
        # Cluster
        clusters = clustering_agent.cluster_data()
        
        if len(clusters) == 0:
            print("\n⚠️ Warning: No clusters formed")
            print("This could mean posts are too spread out spatially or temporally")
            return
        
        # STEP 4: Fire Analysis
        print("\n[AGENT 4/5] FIRE ANALYST")
        fire_analyst = FireAnalystAgent(clustering_agent)
        cluster_reports = fire_analyst.analyze_all_clusters()
        
        # STEP 5: Visualization
        print("\n[AGENT 5/5] VISUALIZATION")
        visualization_agent = VisualizationAgent(fire_analyst)
        map_file = visualization_agent.create_interactive_map()
        
        # Save cluster reports to JSON
        save_cluster_reports(fire_analyst)
        
        # CLEANUP: Delete raw data
        cleanup_raw_data(data_collector)
        
        # Final summary
        print("\n" + "="*80)
        print("PIPELINE COMPLETE!")
        print("="*80)
        print(f"\n✓ Fire: {fire_info['fire_name']}")
        print(f"✓ Clusters analyzed: {len(clusters)}")
        print(f"✓ Map created: {map_file}")
        print(f"✓ Reports saved: cluster_reports.json")
        print(f"\n📂 Output files:")
        print(f"   - {map_file}")
        print(f"   - cluster_reports.json")
        print(f"\n🔒 Privacy: All raw social media data has been deleted")
        print(f"\n🗺️  Open {map_file} in your browser to view the interactive map!")
        print("="*80)
        
    except KeyboardInterrupt:
        print("\n\n⚠️ Pipeline interrupted by user")
        
    except Exception as e:
        print(f"\n\n✗ Pipeline error: {e}")
        import traceback
        traceback.print_exc()
        print("\nPartial results may be available")

# Run the pipeline
if __name__ == "__main__":
    run_wildfire_analysis_pipeline()

print("\n✓ Pipeline ready to run!")
print("Execute: run_wildfire_analysis_pipeline()")